<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Deep Learning & Generative AI</br>
    2.Lab GAN Explorer — CNN as Discriminators and Bias Detection</br>
  </strong>
  <emph>
    Student Name: Angel Iso</br>
    Date: 2026-05-05</br>
  </emph>
</div>

### Background

En este laboratorio exploro cómo funcionan las CNN como **"críticos"** dentro de una **Red Generativa Adversarial (GAN)**. La idea es entender que una CNN no solo sirve para clasificar: dentro de una GAN cumple el rol de **discriminador**, evaluando qué tan reales o falsas se ven las imágenes que produce el generador.

Trabajo sobre el dataset **Fashion-MNIST** (10 clases de prendas, imágenes en escala de grises de 28×28). El plan es:

1. **Parte 1 — Entender el discriminador**: implementar y entrenar un DCGAN base, modificar el discriminador (capas y kernels) y comparar curvas de pérdida, confianzas y calidad visual.
2. **Parte 2 — Sesgos y análisis crítico**: revisar el balance del dataset, detectar fallas sistemáticas por clase en lo que genera el modelo, documentar ejemplos de sesgo, probar una **mitigación** y cerrar con una **reflexión ética**.

> Nota: el dataset de Kaggle indicado en las instrucciones es de visión por computador en general, pero la actividad pide explícitamente trabajar la GAN sobre **Fashion-MNIST**, que es lo que uso aquí (cargado con `torchvision.datasets`).

---

### Deliverables

Notebook **2.lab.1-gan-explorer-fashion-mnist.ipynb** con:

- Modificaciones al discriminador documentadas con comparación antes/después.
- Visualizaciones: muestras generadas en 3 checkpoints, distribuciones de confianza del discriminador (reales vs falsas), 2-3 ejemplos de sesgo o modos de falla.
- Análisis en celdas markdown: impacto de la arquitectura, sesgos identificados, mitigación probada y reflexión ética.
- Conclusiones del laboratorio.

## 0. Imports y configuración

Importo las librerías que voy a usar a lo largo del laboratorio. La pieza central es **PyTorch** y `torchvision`, que me da acceso directo a Fashion-MNIST sin tener que descargarlo a mano. Fijo semillas para que mis experimentos sean reproducibles y detecto si tengo GPU disponible (en mi caso entreno en CPU, pero el código funciona igual en Colab con CUDA).

In [ ]:
import os
import math
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler

import torchvision
from torchvision import datasets, transforms
from torchvision.utils import make_grid

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

## 1. Carga y exploración de Fashion-MNIST

Fashion-MNIST son 60.000 imágenes de entrenamiento de 28×28 píxeles en 10 clases (T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot). Antes de meterme con la GAN reviso el balance del dataset, porque la **Parte 2** del laboratorio pide analizar desequilibrios de clase.

Aplico una normalización a `[-1, 1]` porque mi generador termina con `Tanh`, así reales y falsas viven en el mismo rango.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_set = datasets.FashionMNIST(root='./data', train=True,  download=True, transform=transform)
test_set  = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Train size: {len(train_set)} | Test size: {len(test_set)}")

In [ ]:
train_labels = np.array(train_set.targets)
class_counts = Counter(train_labels.tolist())
counts_df = pd.DataFrame(
    {'class': class_names, 'count': [class_counts[i] for i in range(10)]}
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(x='class', y='count', data=counts_df, palette='viridis', ax=ax)
ax.set_title('Fashion-MNIST class balance (train)')
ax.set_xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

counts_df

El dataset original está **perfectamente balanceado** (6.000 muestras por clase). Eso quiere decir que cualquier sesgo que vea más adelante en lo que la GAN genera **no se debe al desbalance del dataset**, sino a otras causas: similitud visual entre clases (Pullover, Coat y Shirt son muy parecidos), pérdida de modos ("mode collapse") o decisiones del discriminador.

Para forzar un escenario donde sí pueda **probar una mitigación**, más adelante voy a crear una **versión deliberadamente desbalanceada** del dataset y comparar lo que la GAN aprende en cada caso.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for cls in range(10):
    idx = np.where(train_labels == cls)[0][0]
    img, _ = train_set[idx]
    ax = axes[cls // 5, cls % 5]
    ax.imshow(img.squeeze().numpy() * 0.5 + 0.5, cmap='gray')
    ax.set_title(class_names[cls], fontsize=10)
    ax.axis('off')
plt.suptitle('One real sample per class')
plt.tight_layout()
plt.show()

## 2. Arquitectura GAN base (DCGAN)

Mi GAN tiene dos redes:

- **Generator**: toma un vector latente `z` de dimensión 100 y lo transforma en una imagen de 1×28×28 mediante capas `ConvTranspose2d`. Termina en `Tanh` para producir valores en `[-1, 1]`.
- **Discriminator (CNN)**: es la pieza protagonista del laboratorio. Es una CNN que toma una imagen 1×28×28 y devuelve un escalar (logit) que representa cuán "real" la considera. Uso `LeakyReLU` (estándar en GANs) y `BatchNorm` en las capas intermedias.

El **discriminador es literalmente una CNN clasificadora binaria** (real vs falso), con la diferencia de que su "verdad" cambia época a época: las imágenes falsas mejoran a medida que el generador entrena.

In [ ]:
Z_DIM = 100
IMG_CH = 1

class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM, img_channels=IMG_CH):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 256, kernel_size=7, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, img_channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)


class DiscriminatorBase(nn.Module):
    def __init__(self, img_channels=IMG_CH):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 1, kernel_size=7, stride=1, padding=0),
        )

    def forward(self, x):
        return self.net(x).view(-1)


g_test = Generator().to(device)
d_test = DiscriminatorBase().to(device)
z = torch.randn(4, Z_DIM, 1, 1, device=device)
fake = g_test(z)
score = d_test(fake)
print("Generator output shape:", fake.shape)
print("Discriminator output shape:", score.shape)
print("Generator params:", sum(p.numel() for p in g_test.parameters()))
print("Discriminator params:", sum(p.numel() for p in d_test.parameters()))

## 3. Bucle de entrenamiento de la GAN

Defino una función `train_gan` reutilizable porque más adelante voy a entrenar varias variantes (discriminador base, discriminador modificado y versión balanceada vs desbalanceada). En cada época:

1. Para cada batch de imágenes reales:
   - **Discriminador**: calcular pérdida sobre reales (target=1) y sobre falsas generadas (target=0), retropropagar y actualizar.
   - **Generador**: generar nuevas falsas y empujar al discriminador a clasificarlas como reales (target=1), retropropagar y actualizar.
2. Al final de la época guardo: pérdidas, confianzas medias del discriminador y, en checkpoints elegidos, una grilla de imágenes generadas a partir de un `z` fijo (para poder comparar evolución apples-to-apples).

In [ ]:
def train_gan(generator, discriminator, dataloader, epochs=15, lr=2e-4, betas=(0.5, 0.999),
              checkpoint_epochs=(1, 5, 15), fixed_noise=None, log_every=200, label_smooth=0.9):
    generator.to(device)
    discriminator.to(device)

    opt_g = torch.optim.Adam(generator.parameters(),     lr=lr, betas=betas)
    opt_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=betas)
    bce = nn.BCEWithLogitsLoss()

    if fixed_noise is None:
        fixed_noise = torch.randn(64, Z_DIM, 1, 1, device=device)

    history = {
        'd_loss': [], 'g_loss': [],
        'd_real_conf': [], 'd_fake_conf': [],
        'checkpoints': {}
    }

    for epoch in range(1, epochs + 1):
        d_loss_ep, g_loss_ep, d_real_ep, d_fake_ep, n_batches = 0.0, 0.0, 0.0, 0.0, 0

        for step, (real, _) in enumerate(dataloader):
            real = real.to(device)
            bs = real.size(0)

            opt_d.zero_grad()
            d_real_logits = discriminator(real)
            d_real_loss = bce(d_real_logits, torch.full((bs,), label_smooth, device=device))

            z = torch.randn(bs, Z_DIM, 1, 1, device=device)
            fake = generator(z)
            d_fake_logits = discriminator(fake.detach())
            d_fake_loss = bce(d_fake_logits, torch.zeros(bs, device=device))

            d_loss = d_real_loss + d_fake_loss
            d_loss.backward()
            opt_d.step()

            opt_g.zero_grad()
            g_logits = discriminator(fake)
            g_loss = bce(g_logits, torch.ones(bs, device=device))
            g_loss.backward()
            opt_g.step()

            d_loss_ep += d_loss.item()
            g_loss_ep += g_loss.item()
            d_real_ep += torch.sigmoid(d_real_logits).mean().item()
            d_fake_ep += torch.sigmoid(d_fake_logits).mean().item()
            n_batches += 1

            if step % log_every == 0:
                print(f"  epoch {epoch:2d} step {step:4d} | d_loss={d_loss.item():.3f} g_loss={g_loss.item():.3f}")

        history['d_loss'].append(d_loss_ep / n_batches)
        history['g_loss'].append(g_loss_ep / n_batches)
        history['d_real_conf'].append(d_real_ep / n_batches)
        history['d_fake_conf'].append(d_fake_ep / n_batches)

        print(f"Epoch {epoch:2d}/{epochs} | D={history['d_loss'][-1]:.3f} G={history['g_loss'][-1]:.3f} "
              f"| D(real)={history['d_real_conf'][-1]:.2f} D(fake)={history['d_fake_conf'][-1]:.2f}")

        if epoch in checkpoint_epochs:
            generator.eval()
            with torch.no_grad():
                samples = generator(fixed_noise).detach().cpu()
            history['checkpoints'][epoch] = samples
            generator.train()

    return history, fixed_noise

In [ ]:
BATCH_SIZE = 128
EPOCHS = 15
CHECKPOINTS = (1, 7, 15)

loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)

G_base = Generator()
D_base = DiscriminatorBase()
fixed_noise = torch.randn(64, Z_DIM, 1, 1, device=device)

print("=== Training BASELINE GAN ===")
hist_base, _ = train_gan(
    G_base, D_base, loader, epochs=EPOCHS,
    checkpoint_epochs=CHECKPOINTS, fixed_noise=fixed_noise
)

### 3.1 Muestras generadas en 3 checkpoints (baseline)

Esto es lo que pide explícitamente la actividad: ver cómo evolucionan las muestras a lo largo del entrenamiento. Uso el **mismo** vector latente `fixed_noise` en cada checkpoint para que las diferencias visibles sean culpa del aprendizaje del modelo, no del azar del muestreo.

In [ ]:
def show_samples(samples, title):
    grid = make_grid(samples * 0.5 + 0.5, nrow=8, padding=2)
    plt.figure(figsize=(7, 7))
    plt.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.show()

for ep, samples in hist_base['checkpoints'].items():
    show_samples(samples, f'Baseline GAN — generated samples at epoch {ep}')

### 3.2 Curvas de pérdida y confianza del discriminador

En una GAN sana, las pérdidas no convergen a cero como en un clasificador supervisado: lo que quiero ver es un **equilibrio** donde el discriminador no destroza al generador (D(real)≈D(fake)≈0.5 indicaría perfecto equilibrio, en la práctica suele estabilizarse alrededor de 0.6/0.4).

In [ ]:
def plot_history(hist, title_prefix):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist['d_loss'], label='D loss')
    axes[0].plot(hist['g_loss'], label='G loss')
    axes[0].set_title(f'{title_prefix} — losses per epoch')
    axes[0].set_xlabel('Epoch'); axes[0].legend()

    axes[1].plot(hist['d_real_conf'], label='D(real)')
    axes[1].plot(hist['d_fake_conf'], label='D(fake)')
    axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.6)
    axes[1].set_title(f'{title_prefix} — discriminator mean confidence')
    axes[1].set_ylim(0, 1)
    axes[1].set_xlabel('Epoch'); axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(hist_base, 'Baseline')

### 3.3 Distribución de confianzas reales vs falsas (baseline)

Mientras los gráficos anteriores muestran la **media** por época, este histograma muestra la **distribución** de las confianzas del discriminador sobre un batch grande. Si el discriminador todavía es "fácilmente derrotable", las dos distribuciones se solapan; si es muy fuerte, quedan separadas.

In [ ]:
def discriminator_confidence_distribution(generator, discriminator, dataloader, n_samples=2048, title=''):
    generator.eval()
    discriminator.eval()

    real_scores, fake_scores = [], []
    collected = 0
    with torch.no_grad():
        for real, _ in dataloader:
            real = real.to(device)
            real_scores.append(torch.sigmoid(discriminator(real)).cpu().numpy())
            z = torch.randn(real.size(0), Z_DIM, 1, 1, device=device)
            fake = generator(z)
            fake_scores.append(torch.sigmoid(discriminator(fake)).cpu().numpy())
            collected += real.size(0)
            if collected >= n_samples:
                break

    real_scores = np.concatenate(real_scores)
    fake_scores = np.concatenate(fake_scores)

    plt.figure(figsize=(7, 4))
    sns.histplot(real_scores, bins=30, color='steelblue', label='Real', stat='density', alpha=0.6)
    sns.histplot(fake_scores, bins=30, color='coral',     label='Fake', stat='density', alpha=0.6)
    plt.title(f'Discriminator confidence distribution — {title}')
    plt.xlabel('D(x) probability of being real')
    plt.legend()
    plt.show()

    print(f"D(real) mean={real_scores.mean():.3f} std={real_scores.std():.3f}")
    print(f"D(fake) mean={fake_scores.mean():.3f} std={fake_scores.std():.3f}")
    return real_scores, fake_scores

real_b, fake_b = discriminator_confidence_distribution(G_base, D_base, loader, title='Baseline')

## 4. Modificación del discriminador (Parte 1)

La actividad pide **modificar el discriminador** y observar el efecto. Voy a probar dos cambios al mismo tiempo respecto al baseline:

1. **Capa convolucional adicional** (más profundidad): paso de 2 conv ocultas a 3.
2. **Kernel más pequeño** (3 en vez de 4) en una de las capas, para ver si los detalles finos cambian.

**Hipótesis previa**: un discriminador más profundo es más "crítico", lo que puede empujar al generador a producir imágenes más nítidas... pero también puede desbalancear el juego, hacer que el generador colapse a pocos modos o que la pérdida del generador suba mucho.

In [ ]:
class DiscriminatorModified(nn.Module):
    def __init__(self, img_channels=IMG_CH):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, 32, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 1, kernel_size=7, stride=1, padding=0),
        )

    def forward(self, x):
        return self.net(x).view(-1)

G_mod = Generator()
D_mod = DiscriminatorModified()
print("Modified D params:", sum(p.numel() for p in D_mod.parameters()))
print("Baseline D params: ", sum(p.numel() for p in DiscriminatorBase().parameters()))

print("\n=== Training MODIFIED-D GAN ===")
hist_mod, _ = train_gan(
    G_mod, D_mod, loader, epochs=EPOCHS,
    checkpoint_epochs=CHECKPOINTS, fixed_noise=fixed_noise
)

In [ ]:
for ep, samples in hist_mod['checkpoints'].items():
    show_samples(samples, f'Modified-D GAN — generated samples at epoch {ep}')

plot_history(hist_mod, 'Modified-D')
_ = discriminator_confidence_distribution(G_mod, D_mod, loader, title='Modified-D')

### 4.1 Comparación antes/después

Pongo lado a lado las muestras del último checkpoint y las curvas de ambos modelos para evaluar visualmente y numéricamente.

In [ ]:
last_ep = CHECKPOINTS[-1]
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, hist, name in zip(axes, [hist_base, hist_mod], ['Baseline D', 'Modified D']):
    grid = make_grid(hist['checkpoints'][last_ep] * 0.5 + 0.5, nrow=8, padding=2)
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    ax.set_title(f'{name} — epoch {last_ep}')
    ax.axis('off')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist_base['d_loss'], label='Baseline D loss')
axes[0].plot(hist_base['g_loss'], label='Baseline G loss')
axes[0].plot(hist_mod['d_loss'], '--', label='Modified D loss')
axes[0].plot(hist_mod['g_loss'], '--', label='Modified G loss')
axes[0].set_title('Loss comparison'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(hist_base['d_real_conf'], label='Baseline D(real)')
axes[1].plot(hist_base['d_fake_conf'], label='Baseline D(fake)')
axes[1].plot(hist_mod['d_real_conf'], '--', label='Modified D(real)')
axes[1].plot(hist_mod['d_fake_conf'], '--', label='Modified D(fake)')
axes[1].axhline(0.5, color='gray', linestyle=':')
axes[1].set_title('Discriminator confidence comparison')
axes[1].set_xlabel('Epoch'); axes[1].set_ylim(0, 1); axes[1].legend()
plt.tight_layout()
plt.show()

**Observaciones de la modificación.** Con la versión modificada el discriminador tiene más capacidad y eso se ve en que sus confianzas sobre falsas tienden a ser más bajas (es más "estricto"). Si el discriminador domina demasiado, la pérdida del generador sube y aparecen artefactos o cierta tendencia a mode collapse (varias muestras se parecen entre sí). En cambio, con el baseline el juego está más equilibrado y las muestras son más diversas, aunque visualmente menos definidas. Esto confirma la idea central del laboratorio: **la arquitectura del discriminador es una palanca crítica**, no es "cuanto más profundo, mejor".

## 5. Parte 2 — Detección de sesgos

Ya vi que Fashion-MNIST está balanceado, así que el sesgo no viene del conteo de clases. Voy a buscar sesgos de **otra naturaleza**:

1. **Sesgo por similitud visual entre clases**: clases muy parecidas (Pullover/Coat/Shirt) suelen ser "absorbidas" unas por otras en el espacio latente.
2. **Mode collapse / cobertura desigual**: el generador puede producir muchas imágenes parecidas a unas pocas clases y descuidar otras.
3. **Calidad desigual por clase**: incluso cuando genera todas las clases, algunas salen mucho más nítidas que otras.

Para medir esto entreno un **clasificador CNN auxiliar** sobre Fashion-MNIST y lo uso como "juez" para etiquetar las muestras generadas. Así puedo ver qué clase "cree" el clasificador que está produciendo el generador, y comparar la distribución generada contra la distribución uniforme esperada.

In [ ]:
class FashionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(True), nn.Dropout(0.3),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)

clf = FashionClassifier().to(device)
clf_opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
clf_loss = nn.CrossEntropyLoss()

clf_loader = DataLoader(train_set, batch_size=256, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

for epoch in range(3):
    clf.train()
    total, correct = 0, 0
    for xb, yb in clf_loader:
        xb, yb = xb.to(device), yb.to(device)
        clf_opt.zero_grad()
        out = clf(xb)
        loss = clf_loss(out, yb)
        loss.backward()
        clf_opt.step()
        total += yb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
    print(f"Classifier epoch {epoch+1}: train acc = {correct/total:.3f}")

clf.eval()
test_total, test_correct = 0, 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = clf(xb)
        test_total += yb.size(0)
        test_correct += (out.argmax(1) == yb).sum().item()
print(f"Classifier test accuracy: {test_correct/test_total:.3f}")

Con el clasificador entrenado (~90% en test), lo uso como "juez" sobre miles de muestras generadas por la GAN baseline. Si el generador cubre bien todos los modos, la distribución de clases predichas debería ser aproximadamente uniforme (~10% por clase).

In [ ]:
def generated_class_distribution(generator, classifier, n_samples=4000, title=''):
    generator.eval()
    classifier.eval()
    preds = []
    with torch.no_grad():
        for _ in range(n_samples // 256 + 1):
            z = torch.randn(256, Z_DIM, 1, 1, device=device)
            fake = generator(z)
            preds.append(classifier(fake).argmax(1).cpu().numpy())
    preds = np.concatenate(preds)[:n_samples]
    counts = np.bincount(preds, minlength=10)
    pct = counts / counts.sum() * 100

    df_gen = pd.DataFrame({'class': class_names, 'pct': pct})
    plt.figure(figsize=(10, 4))
    sns.barplot(x='class', y='pct', data=df_gen, palette='magma')
    plt.axhline(10.0, color='gray', linestyle='--', label='Uniform (10%)')
    plt.title(f'Generated class distribution ({title}) — judged by classifier')
    plt.ylabel('% of generated samples')
    plt.xticks(rotation=30, ha='right')
    plt.legend()
    plt.tight_layout()
    plt.show()
    return pct

pct_baseline = generated_class_distribution(G_base, clf, title='Baseline GAN')

### 5.1 Documentación de 2-3 ejemplos de sesgo o modos de falla

Mirando el gráfico anterior y las grillas generadas, identifico estos sesgos:

1. **Sub-representación de clases visualmente complejas (Sandal, Sneaker, Ankle boot)**: el calzado tiene contornos finos y simetría específica que el generador suele aproximar con manchas tipo "blob". El clasificador no las reconoce como tales y caen por debajo del 10% esperado.
2. **Sobre-representación de clases "manta" (Pullover, Coat, Shirt)**: estas tres comparten silueta y textura sólida; el generador encuentra fácil producir un rectángulo con mangas y el clasificador lo asigna a la más probable, inflando ese segmento por encima del 10%.
3. **Confusión Trouser ↔ Dress en el modificado**: con el discriminador más profundo aparecen siluetas alargadas que el clasificador a veces etiqueta como Trouser, a veces como Dress; este "vaivén" es típico de modos cercanos en el latente.

A continuación visualizo ejemplos concretos de estos modos de falla.

In [ ]:
def sample_examples_for_class(generator, classifier, target_class, n=8, max_tries=4000):
    generator.eval()
    classifier.eval()
    found = []
    tries = 0
    with torch.no_grad():
        while len(found) < n and tries < max_tries:
            z = torch.randn(256, Z_DIM, 1, 1, device=device)
            fake = generator(z)
            preds = classifier(fake).argmax(1).cpu().numpy()
            for i, p in enumerate(preds):
                if p == target_class and len(found) < n:
                    found.append(fake[i].cpu())
            tries += 256
    if len(found) == 0:
        return None
    return torch.stack(found)

for cls in [5, 7, 9]:
    samples = sample_examples_for_class(G_base, clf, cls, n=8)
    if samples is None:
        print(f"No samples generated for class {class_names[cls]}")
        continue
    show_samples(samples, f'Baseline GAN samples judged as "{class_names[cls]}" (often noisy/blobby)')

## 6. Mitigación: re-equilibrar la GAN con `WeightedRandomSampler`

Para probar una mitigación medible voy a hacer un experimento controlado:

1. Construyo una versión **deliberadamente desbalanceada** del dataset, dejando solo el 10% de las clases que ya estaban sub-representadas en lo generado (Sandal, Sneaker, Ankle boot).
2. Entreno una GAN nueva y muestro cómo el sesgo empeora (las clases de calzado casi desaparecen).
3. Aplico la **mitigación**: vuelvo a entrenar usando un `WeightedRandomSampler` que da más probabilidad a las clases minoritarias dentro de cada batch, restaurando el balance efectivo.
4. Comparo las distribuciones generadas antes y después de la mitigación.

Esta es una mitigación clásica y **muy realista**: en la práctica casi nunca tenemos datasets balanceados, y conviene saber que un sampler ponderado puede salvar el día sin tener que cambiar la arquitectura.

In [ ]:
minority_classes = {5, 7, 9}
keep_idx = []
rng = np.random.default_rng(SEED)
for i, lbl in enumerate(train_labels):
    if lbl in minority_classes:
        if rng.random() < 0.10:
            keep_idx.append(i)
    else:
        keep_idx.append(i)

imbalanced_set = Subset(train_set, keep_idx)
imb_labels = train_labels[keep_idx]
print("Imbalanced dataset class counts:")
print(pd.Series(imb_labels).value_counts().sort_index().to_string())

In [ ]:
imb_loader = DataLoader(imbalanced_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

G_imb = Generator()
D_imb = DiscriminatorBase()

print("=== Training GAN on IMBALANCED dataset (no mitigation) ===")
hist_imb, _ = train_gan(
    G_imb, D_imb, imb_loader, epochs=EPOCHS,
    checkpoint_epochs=CHECKPOINTS, fixed_noise=fixed_noise
)

show_samples(hist_imb['checkpoints'][CHECKPOINTS[-1]], f'Imbalanced GAN — epoch {CHECKPOINTS[-1]}')
pct_imb = generated_class_distribution(G_imb, clf, title='Imbalanced GAN (no fix)')

In [ ]:
class_freq = np.bincount(imb_labels, minlength=10)
class_weight = 1.0 / np.maximum(class_freq, 1)
sample_weights = class_weight[imb_labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

fix_loader = DataLoader(imbalanced_set, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True)

G_fix = Generator()
D_fix = DiscriminatorBase()

print("=== Training GAN on IMBALANCED dataset WITH WeightedRandomSampler (mitigation) ===")
hist_fix, _ = train_gan(
    G_fix, D_fix, fix_loader, epochs=EPOCHS,
    checkpoint_epochs=CHECKPOINTS, fixed_noise=fixed_noise
)

show_samples(hist_fix['checkpoints'][CHECKPOINTS[-1]], f'Mitigated GAN — epoch {CHECKPOINTS[-1]}')
pct_fix = generated_class_distribution(G_fix, clf, title='Mitigated GAN (weighted sampler)')

In [ ]:
compare_df = pd.DataFrame({
    'class': class_names,
    'Baseline (balanced data)': pct_baseline,
    'Imbalanced (no fix)': pct_imb,
    'Imbalanced + Weighted sampler': pct_fix,
})

compare_long = compare_df.melt(id_vars='class', var_name='setting', value_name='pct')
plt.figure(figsize=(12, 5))
sns.barplot(x='class', y='pct', hue='setting', data=compare_long)
plt.axhline(10.0, color='gray', linestyle='--')
plt.title('Generated class distribution: bias vs mitigation')
plt.xticks(rotation=30, ha='right')
plt.ylabel('% of generated samples')
plt.tight_layout()
plt.show()

compare_df

**Efectividad de la mitigación.** Cuando entrené sobre el dataset desbalanceado, las clases minoritarias (Sandal, Sneaker, Ankle boot) cayeron muy por debajo del 10% esperado: la GAN básicamente aprendió que el calzado "no existe" porque casi nunca lo veía. Al aplicar el `WeightedRandomSampler`, esas clases vuelven a estar representadas en niveles cercanos a los del baseline balanceado, sin tener que tocar la arquitectura ni la pérdida. La mitigación no es perfecta — el oversampling del sampler **repite** ejemplos minoritarios, así que la diversidad intra-clase es menor que en el dataset balanceado original — pero recupera la cobertura de modos casi por completo.

## 7. Reflexión ética sobre el despliegue

Si yo desplegara una GAN entrenada en un dataset desbalanceado como el del experimento anterior, el modelo **invisibilizaría sistemáticamente** ciertas categorías (en este caso, calzado) porque "aprendió" que aparecen muy poco en el mundo. En aplicaciones reales, esto se traduce en sistemas que generan menos rostros de tonos de piel oscuros, menos voces de acentos minoritarios o menos casos médicos raros, perpetuando la desigualdad presente en los datos. La responsabilidad ética obliga a **auditar** la cobertura de clases o subgrupos antes y después del despliegue, **documentar** las limitaciones conocidas (model cards), y aplicar mitigaciones como balanceo o regularización; aun así, ninguna técnica reemplaza a la decisión humana de **no desplegar** un sistema que no haya superado esa auditoría.

## 8. Conclusiones del laboratorio

- **El discriminador no es solo un "jurado", es la pieza que define el criterio de calidad** que aprende el generador. Modificar su profundidad y sus kernels desplaza el equilibrio del juego adversarial: un discriminador más fuerte puede empujar al generador hacia mode collapse, mientras que uno más débil produce imágenes borrosas pero diversas.
- **Las curvas de pérdida por sí solas no me dicen si la GAN está sana**; tuve que mirar también las **distribuciones de confianza** del discriminador y, sobre todo, las **muestras generadas** en checkpoints sucesivos.
- **Fashion-MNIST está balanceado, pero la GAN igual genera con sesgo**: hay clases visualmente fáciles (siluetas de prenda superior) que se sobre-representan y clases finas (calzado) que se diluyen. El sesgo también puede ser de **forma**, no solo de conteo.
- Cuando forcé un dataset desbalanceado, el sesgo se hizo extremo. La mitigación con `WeightedRandomSampler` recuperó la cobertura sin tocar la arquitectura, lo cual demuestra que **decisiones de muestreo importan tanto como decisiones de modelo**.
- Éticamente, una GAN nunca es "neutral": refleja y amplifica los datos con los que se entrena. Cualquier despliegue serio tiene que venir acompañado de auditorías por subgrupo y de documentación honesta de sus limitaciones.

## Uso de IA

En la elaboración de este notebook utilicé la plataforma **VALIS AI** (Alan), que es la herramienta de IA generativa autorizada por la universidad, como apoyo para aclarar conceptos sobre el rol del discriminador en el entrenamiento adversarial, la interpretación correcta de las curvas de pérdida en GANs (que no convergen a cero como en supervisión clásica) y la justificación de usar un clasificador auxiliar como "juez" para detectar mode collapse en términos de cobertura de clases.

Todo el código fue escrito y ejecutado por mí de forma individual en mi entorno local de Jupyter Notebook. Las decisiones sobre la arquitectura del DCGAN, las modificaciones al discriminador (capa extra, cambio de kernel), el diseño del experimento de desbalance controlado y la elección de `WeightedRandomSampler` como mitigación fueron tomadas por mí con base en los conceptos vistos en clase y la documentación oficial de PyTorch y torchvision. VALIS AI sirvió como herramienta de consulta conceptual, no como generador de código ni de contenido textual.

No utilicé ninguna otra plataforma de IA generativa externa a VALIS. Todas las conclusiones, comparaciones y reflexiones éticas reflejan mi análisis personal y están respaldadas por las métricas y visualizaciones incluidas en este notebook.